# 2025 Unique Names
Thought this would be useful for tracking how many times someone's name appears
Additionally, tracking how many months a person receives support

In [54]:
import pandas as pd
from enum import Enum

class Date(Enum): # Enumeration class to assign the month numbers to month strings. Easier to maintain human readability
    Jan = 1
    Feb = 2
    Mar = 3
    Apr = 4
    May = 5
    Jun = 6
    Jul = 7
    Aug = 8
    Sep = 9
    Oct = 10
    Nov = 11
    Dec = 12

df = pd.read_csv("../Data/PantrytoPorchData2025.csv")
df.drop(columns=[" Volunteer Name", "Pantry", "Adults", "Children", "Total in house"], inplace=True)
df.head(10)
temp_dict = {
    "name": [],
    "delivery count": [],
    "months supported": [],
    "month started": [],
    "last supported": []
    # if someone received deliveries in December it's not clear if they still are receving them into the new year
}
for row in df.itertuples():
    # print(row)
    cur_name = row.Name
    date_num = int(row.Date.split("/")[0])
    cur_date = Date(date_num).name
    
    if cur_name not in temp_dict["name"]:
        temp_dict["name"].append(cur_name)
        temp_dict["delivery count"].append(1)
        temp_dict["months supported"].append([cur_date])
        temp_dict["last supported"].append(cur_date)

        if row[3] == 1:
            temp_dict["month started"].append(cur_date)

        else:
            temp_dict["month started"].append(pd.NA) # This is the normal format for NA value, it does look ugly in the csv though
            # I think its good to track when someone started, but not everyone started this year. Important to keep this information to avoid misrepresentation


    elif cur_name in temp_dict["name"]:
        name_ind = temp_dict["name"].index(cur_name)
        temp_dict["delivery count"][name_ind] += 1

        if temp_dict["last supported"][name_ind] != cur_date:
            temp_dict["months supported"][name_ind].append(cur_date)
            temp_dict["last supported"][name_ind] = cur_date

df = pd.DataFrame(dict(temp_dict))
df.head()
df.to_csv("../Data/Leo2025.csv", index= False)


# New Clients vs Returning
Overlapping barchart or line?
Mainly worried about content, can make it prettier later

In [ ]:
import matplotlib.pyplot as plt

new_or_return = {
    "Jan": {"new": 0, "returning": 0},
    "Feb": {"new": 0, "returning": 0},
    "Mar": {"new": 0, "returning": 0},
    "Apr": {"new": 0, "returning": 0},
    "May": {"new": 0, "returning": 0},
    "Jun": {"new": 0, "returning": 0},
    "Jul": {"new": 0, "returning": 0},
    "Aug": {"new": 0, "returning": 0},
    "Sep": {"new": 0, "returning": 0},
    "Oct": {"new": 0, "returning": 0},
    "Nov": {"new": 0, "returning": 0},
    "Dec": {"new": 0, "returning": 0}
}

def increment_from_month(supported_months):
    # Take a list of months, incrementing returning count for each one
    for month in supported_months:
        new_or_return[month]["returning"] += 1

for row in df.itertuples():
    start_date = row[4]
    months_supported: list = row[3].copy()
    if pd.isna(start_date):
        increment_from_month(months_supported)

    else:
        new_or_return[start_date]["new"] += 1 # increase the count of new user
        months_supported.remove(start_date) # don't want to double count the start month
        increment_from_month(months_supported)

new_or_return_df = pd.DataFrame(dict(new_or_return))
new_or_return_df.to_csv("../Data/NeworReturning.csv")

# fig, ax = plt.subplot()